# Stochastic White Noise Analysis (SWNA) with Memory
### Workshop — Hands-on Session

**Package:** `whitenoise` — SWNA with Memory framework (Bernido & Carpio-Bernido)

---

## The Full Pipeline

| Step | Function | What it does |
|---|---|---|
| A | `wn.read_csv()` | Load CSV → time, values, metadata |
| B | `wn.plot_series()` | Plot the raw time series |
| C | `wn.compute_msd()` | Compute empirical MSD |
| D | `wn.plot_msd_raw()` | Inspect the MSD shape |
| E | `wn.fit_msd()` | Fit model → extract μ, R², 95% CIs |
| F | `fit.summary()` | Print the full parameter table |
| G | *(matplotlib)* | Overlay fitted curve on empirical MSD |
| H | *(matplotlib)* | Plot displacement PDF vs theoretical Gaussian |
| I | `wn.analyze()` | Run steps A–H automatically in one call |

---

**The memory parameter μ:**

| μ | Regime | Meaning |
|---|---|---|
| < 0.95 | Subdiffusive | Past fluctuations resist change |
| 0.95–1.05 | Near-Brownian | Weak or no memory |
| 1.05–2.0 | Superdiffusive | Past fluctuations reinforce future ones |
| > 2.0 | Hyperballistic | Strongly persistent dynamics |

---
## Setup — Install and import
Run once at the start of every session.

In [ ]:
!pip install git+https://github.com/pnayga/whitenoise.git -q
print('Installation complete.')

In [ ]:
import whitenoise as wn
import matplotlib.pyplot as plt
import numpy as np

print(f'whitenoise {wn.__version__} ready.')

---
## Available models

The package implements 16 models from Table 3.1 of Bernido & Carpio-Bernido (2015).
**You choose the model — the package never auto-selects.**

In [ ]:
wn.list_models()

---
## CSV format

```
time [months], sunspot_number [count]
1, 12.4
2, 15.1
3, 14.8
```

- **Row 1** → header (required)
- **Column 1** → time or index (independent variable)
- **Column 2** → your observable
- Format: `name [unit]` — unit in square brackets
- Unitless: `flux []`

---
## Demo datasets

Four sample datasets — one per implemented model, each based on a published physical system:

| File | Physical system | Model | Expected μ | Reference |
|---|---|---|---|---|
| `exponential_sunspot.csv` | Solar sunspot numbers | `exponential` | ~1.12 | Toledo et al. 2024 |
| `cosine_co2.csv` | CO₂ concentration | `cosine` | ~0.93 | Elnar et al. 2024 |
| `cosine_xray.csv` | X-ray binary flux | `cosine` | ~1.22 | Calotes 2024 |
| `sample_series.csv` | Synthetic fBm | `fbm` | H ≈ 0.72 | — |

In [ ]:
base = 'https://raw.githubusercontent.com/pnayga/whitenoise/main/sample_data/'
for fname in ['exponential_sunspot.csv', 'cosine_co2.csv', 'cosine_xray.csv', 'sample_series.csv']:
    !wget -q {base}{fname}
    print(f'Downloaded: {fname}')

---
---
# Step-by-step pipeline demo

We will use **`exponential_sunspot.csv`** and walk through every step manually.

---
## Step A — Load the data
### `wn.read_csv(path)`

Reads the CSV and returns three things:
- **`time`** — Column 1 values (independent variable)
- **`values`** — Column 2 values (your observable)
- **`meta`** — dictionary: column names, units, axis labels, N points

In [ ]:
time, values, meta = wn.read_csv('exponential_sunspot.csv')

print(f"Dataset  : {meta['y_name']}")
print(f"Unit     : {meta['y_unit']}")
print(f"N points : {meta['n_points']}")
print(f"Time     : {time[0]:.0f} to {time[-1]:.0f} ({meta['x_unit']})")
print(f"Range    : {values.min():.2f} to {values.max():.2f} ({meta['y_unit']})")

---
## Step B — Plot the raw time series
### `wn.plot_series(time, values)`

Before any analysis, always look at your data.

Ask yourself:
- Is there a long-term trend?
- Is the signal oscillatory or irregular?
- Are there obvious outliers or gaps?

In [ ]:
wn.plot_series(
    time, values,
    title=meta['y_name'],
    xlabel=meta['x_label'],
    ylabel=meta['y_label'],
)

---
## Step C — Compute the empirical MSD
### `wn.compute_msd(values)`

The **Mean Square Displacement** measures how much the observable typically changes over a lag Δ:

$$\text{MSD}(\Delta) = \frac{1}{N-\Delta} \sum_{i=1}^{N-\Delta} \left[ x(i+\Delta) - x(i) \right]^2$$

- Small Δ → many pairs → **reliable** estimate
- Large Δ → few pairs → **noisier** estimate

The **shape** of the MSD tells you the memory regime before any fitting.

In [ ]:
lags, msd = wn.compute_msd(values)

print(f'Number of lags : {len(lags)}')
print(f'Lag range      : {lags[0]} to {lags[-1]}')
print(f'MSD range      : {msd.min():.4f} to {msd.max():.4f}')

---
## Step D — Plot the empirical MSD
### `wn.plot_msd_raw(lags, msd)`

Inspect the MSD shape **before** choosing a model.

The dashed line is a **linear reference** (Brownian, μ = 1).
- MSD curves **above** the line → superdiffusive
- MSD curves **below** the line → subdiffusive
- MSD follows the line → near-Brownian

In [ ]:
wn.plot_msd_raw(
    lags, msd,
    title=f"Empirical MSD — {meta['y_name']}",
    xlabel=meta['x_label'],
)

---
## Step E — Fit a model to the MSD
### `wn.fit_msd(lags, msd, model)`

Fits **N · msd_model(Δ, params)** to the empirical MSD using least squares.

Returns a **`FitResult`** object containing:
- `fit.params` — fitted values of μ (and ν or β), plus N
- `fit.std_errors` — standard error of each parameter
- `fit.confidence_intervals` — 95% CI for each parameter
- `fit.r_squared` — goodness of fit (R²)
- `fit.lags_used` — which lags were included
- `fit.msd_fitted` — the fitted MSD curve values

**R² guideline:** ≥ 0.8 good &nbsp;·&nbsp; 0.5–0.8 moderate &nbsp;·&nbsp; < 0.5 try another model

In [ ]:
MODEL = 'exponential'   # choose: 'cosine', 'exponential', 'sine', 'fbm'

fit = wn.fit_msd(lags, msd, model=MODEL)

print(f'Model   : {MODEL}')
print(f'R²      : {fit.r_squared:.4f}')
print(f'Params  : {fit.params}')

---
## Step F — Read the parameter table
### `fit.summary()`

Prints all fitted parameters with standard errors and 95% confidence intervals.

**N** is the normalization scalar — it absorbs the amplitude of your data so that μ is scale-independent. Only μ (and ν or β) carry physical meaning.

In [ ]:
print(fit.summary())

---
## Step G — Overlay the fitted curve on the empirical MSD

The fitted curve `fit.msd_fitted` is the model formula evaluated at the fitted parameters:

$$\text{MSD}_{\text{fitted}}(\Delta) = N \cdot \text{msd}_{\text{model}}(\Delta,\, \mu,\, \ldots)$$

Plotting it on top of the empirical MSD shows how well the model captures the data.

In [ ]:
mu_key = 'H' if MODEL == 'fbm' else 'mu'
mu_val = fit.params[mu_key]

fig, ax = plt.subplots(figsize=(7, 4.5))

# Empirical MSD (from Step C)
ax.scatter(lags, msd,
           s=12, color='#555555', alpha=0.6, label='Empirical MSD', zorder=3)

# Fitted curve (from fit.msd_fitted)
finite = np.isfinite(fit.msd_fitted)
ax.plot(fit.lags_used[finite], fit.msd_fitted[finite],
        color='#1B3A6B', lw=2.0,
        label=f'Fitted {MODEL}  (R²={fit.r_squared:.4f})', zorder=4)

ax.set_xlabel(meta['x_label'])
ax.set_ylabel('MSD')
ax.set_title(f"{meta['y_name']} — MSD Fit")
ax.legend()
ax.annotate(f'{mu_key} = {mu_val:.4f}\nR² = {fit.r_squared:.4f}',
            xy=(0.97, 0.05), xycoords='axes fraction',
            ha='right', va='bottom', fontsize=9,
            bbox=dict(facecolor='white', edgecolor='#cccccc', alpha=0.9))
plt.tight_layout()
plt.show()

---
## Step H — Plot the displacement PDF

The SWNA framework predicts that displacements at lag T follow a **Gaussian distribution**:

$$P(\Delta x;\, T) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{\Delta x^2}{2\sigma^2}\right), \qquad \sigma^2 = N \cdot \text{msd}_{\text{model}}(T,\, \mu,\, \ldots)$$

**Displacements** are computed as:
$$\Delta x(i) = x(i + T) - x(i), \qquad i = 1, 2, \ldots, N-T$$

The histogram of these values should match the theoretical curve if the model is correct.

In [ ]:
# Choose lag T (as an index into the lags array)
lag_index = len(lags) // 10
T = int(lags[lag_index])
print(f'Using T = {T}')

# Compute displacements: x(i+T) - x(i)
displacements = values[T:] - values[:-T]
print(f'Number of displacement values: {len(displacements)}')
print(f'Empirical mean : {np.mean(displacements):.4f}')
print(f'Empirical std  : {np.std(displacements):.4f}')

# Theoretical sigma² = N * msd_model(T, mu, ...)
model_info = wn.get_model(MODEL)
phys_names = model_info['params']                      # e.g. ['mu', 'beta']
phys_vals  = [fit.params[p] for p in phys_names]      # e.g. [1.12, 0.10]
sigma2 = model_info['msd'](T, *phys_vals) * fit.params['N']
print(f'Theoretical sigma²: {sigma2:.4f}   sigma: {sigma2**0.5:.4f}')

In [ ]:
mu_disp = np.mean(displacements)   # center at empirical mean

fig, ax = plt.subplots(figsize=(7, 4.5))

# Empirical histogram of displacements
ax.hist(displacements, bins='auto', density=True,
        color='#888888', alpha=0.55, label='Empirical displacements')

# Theoretical Gaussian
dx = np.linspace(displacements.min(), displacements.max(), 500)
pdf = np.exp(-(dx - mu_disp)**2 / (2 * sigma2)) / np.sqrt(2 * np.pi * sigma2)
ax.plot(dx, pdf, color='#C0392B', lw=2.0, label=f'Theoretical PDF  (T={T})')

ax.set_xlabel(f"\u0394{meta['y_label']}")
ax.set_ylabel('Probability density')
ax.set_title(f"{meta['y_name']} \u2014 Displacement PDF at T={T}")
ax.legend()
plt.tight_layout()
plt.show()

---
## Step I — The full pipeline in one call
### `wn.analyze()` + `wn.plot_diagnostics()`

Steps A through H above are exactly what `wn.analyze()` does internally.
Once you understand each step, you can run the entire pipeline in two lines.

In [ ]:
result = wn.analyze('exponential_sunspot.csv', model='exponential')

In [ ]:
result.summary()

In [ ]:
fig = wn.plot_diagnostics(result)
plt.show()

---
## Model comparison — try a different model

If R² is low, try another model. Compare R² values to decide which fits best.
The researcher always chooses — the package never auto-selects.

In [ ]:
result_cos = wn.analyze('exponential_sunspot.csv', model='cosine', verbose=False)

print(f"exponential  R² = {result.fit.r_squared:.4f}   mu = {result.fit.params['mu']:.4f}")
print(f"cosine       R² = {result_cos.fit.r_squared:.4f}   mu = {result_cos.fit.params['mu']:.4f}")
print()
print('Higher R² = better model choice for this dataset')

---
## Optional — Preprocessing before analysis

Use these if your data has a trend or needs to be rescaled **before** computing the MSD.
These are manual steps — the pipeline never calls them automatically.

| Function | What it does |
|---|---|
| `wn.detrend(values, method='linear')` | Subtract a linear (or polynomial) trend |
| `wn.normalize(values, method='zscore')` | Z-score, min-max, or mean normalization |
| `wn.smooth(values, window=5)` | Moving average or Gaussian smoothing |

In [ ]:
time, values, meta = wn.read_csv('exponential_sunspot.csv')

# Remove linear trend first
fluctuations = wn.detrend(values, method='linear')

# Then analyze the residuals
result_detrended = wn.analyze(fluctuations, model='exponential', label='sunspot_detrended')
result_detrended.summary()

---
---
# ✋ Hands-on: Your own data

Repeat the same pipeline on your own CSV file.
Go through **each step** — do not jump straight to `wn.analyze()`.

**Your CSV must follow this format:**
```
time [unit], variable_name [unit]
1, 23.5
2, 24.1
...
```

In [ ]:
from google.colab import files
uploaded = files.upload()
MY_FILE = list(uploaded.keys())[0]
print(f'Uploaded: {MY_FILE}')

### Step A — `wn.read_csv()`

In [ ]:
time, values, meta = wn.read_csv(MY_FILE)

print(f"Dataset  : {meta['y_name']}")
print(f"Unit     : {meta['y_unit']}")
print(f"N points : {meta['n_points']}")
print(f"Time     : {time[0]:.0f} to {time[-1]:.0f} ({meta['x_unit']})")
print(f"Range    : {values.min():.4f} to {values.max():.4f}")

### Step B — `wn.plot_series()`

In [ ]:
wn.plot_series(
    time, values,
    title=meta['y_name'],
    xlabel=meta['x_label'],
    ylabel=meta['y_label'],
)

### Step C — `wn.compute_msd()`

In [ ]:
lags, msd = wn.compute_msd(values)
print(f'Lag range: {lags[0]} to {lags[-1]}   ({len(lags)} lags)')

### Step D — `wn.plot_msd_raw()`

In [ ]:
wn.plot_msd_raw(
    lags, msd,
    title=f"Empirical MSD — {meta['y_name']}",
    xlabel=meta['x_label'],
)

### Step E — `wn.fit_msd()`

Based on the MSD shape you saw in Step D, choose a model:
- **`exponential`** — superdiffusive physical systems (sunspot, earthquake, etc.)
- **`cosine`** — oscillatory or cyclical systems (CO₂, coral, X-ray binaries)
- **`fbm`** — self-similar random processes
- **`sine`** — alternative to cosine

In [ ]:
MY_MODEL = 'cosine'   # <- change this based on your data

fit = wn.fit_msd(lags, msd, model=MY_MODEL)
print(f'Model  : {MY_MODEL}')
print(f'R²     : {fit.r_squared:.4f}')
print(f'Params : {fit.params}')

### Step F — `fit.summary()`

In [ ]:
print(fit.summary())

### Step G — MSD fit plot

In [ ]:
mu_key = 'H' if MY_MODEL == 'fbm' else 'mu'
mu_val = fit.params[mu_key]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(lags, msd, s=12, color='#555555', alpha=0.6, label='Empirical MSD')
finite = np.isfinite(fit.msd_fitted)
ax.plot(fit.lags_used[finite], fit.msd_fitted[finite],
        color='#1B3A6B', lw=2.0, label=f'Fitted {MY_MODEL}  (R²={fit.r_squared:.4f})')
ax.set_xlabel(meta['x_label'])
ax.set_ylabel('MSD')
ax.set_title(f"{meta['y_name']} \u2014 MSD Fit")
ax.legend()
ax.annotate(f'{mu_key} = {mu_val:.4f}\nR\u00b2 = {fit.r_squared:.4f}',
            xy=(0.97, 0.05), xycoords='axes fraction', ha='right', va='bottom', fontsize=9,
            bbox=dict(facecolor='white', edgecolor='#cccccc', alpha=0.9))
plt.tight_layout()
plt.show()

### Step H — Displacement PDF

In [ ]:
lag_index = len(lags) // 10
T = int(lags[lag_index])
displacements = values[T:] - values[:-T]

model_info = wn.get_model(MY_MODEL)
phys_names = model_info['params']
phys_vals  = [fit.params[p] for p in phys_names]
sigma2     = model_info['msd'](T, *phys_vals) * fit.params['N']
mu_disp    = np.mean(displacements)

print(f'T = {T}')
print(f'Empirical std   : {np.std(displacements):.4f}')
print(f'Theoretical std : {sigma2**0.5:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(displacements, bins='auto', density=True,
        color='#888888', alpha=0.55, label='Empirical displacements')
dx = np.linspace(displacements.min(), displacements.max(), 500)
pdf = np.exp(-(dx - mu_disp)**2 / (2 * sigma2)) / np.sqrt(2 * np.pi * sigma2)
ax.plot(dx, pdf, color='#C0392B', lw=2.0, label=f'Theoretical PDF  (T={T})')
ax.set_xlabel(f"\u0394{meta['y_label']}")
ax.set_ylabel('Probability density')
ax.set_title(f"{meta['y_name']} \u2014 Displacement PDF at T={T}")
ax.legend()
plt.tight_layout()
plt.show()

### Step I — Full pipeline with `wn.analyze()`

Now run everything in one call and compare with what you computed manually.

In [ ]:
result = wn.analyze(MY_FILE, model=MY_MODEL)
result.summary()
fig = wn.plot_diagnostics(result)
plt.show()

In [ ]:
wn.export_csv(result, 'my_results.csv')
files.download('my_results.csv')

---
## Bonus — Compare multiple datasets
### `wn.compare()`

In [ ]:
uploaded_multi = files.upload()
file_list = list(uploaded_multi.keys())
print(f'Uploaded {len(file_list)} files: {file_list}')

In [ ]:
comparison = wn.compare(file_list, model=MY_MODEL)
wn.print_comparison(comparison)

In [ ]:
fig = wn.publish_comparison(comparison)
plt.show()

---
## Quick Reference

| Step | Function | Input | Output |
|---|---|---|---|
| A | `wn.read_csv(file)` | CSV path | time, values, meta |
| B | `wn.plot_series(time, values)` | arrays | plot |
| C | `wn.compute_msd(values)` | 1D array | lags, msd |
| D | `wn.plot_msd_raw(lags, msd)` | arrays | plot |
| E | `wn.fit_msd(lags, msd, model)` | arrays + model name | FitResult |
| F | `fit.summary()` | FitResult | parameter table |
| — | `wn.get_model(name)` | model name | model info dict |
| I | `wn.analyze(file, model)` | CSV path | AnalysisResult |
| — | `wn.plot_diagnostics(result)` | AnalysisResult | 4-panel figure |
| — | `wn.compare(files, model)` | list of paths | ComparisonResult |
| — | `wn.export_csv(result, path)` | AnalysisResult | CSV file |
| — | `wn.list_models()` | — | model table |

**Preprocessing:** `wn.detrend()` · `wn.normalize()` · `wn.smooth()`

**Models:** `cosine` · `exponential` · `sine` · `fbm`